In [ ]:
# With much thanks to Islam S. for identifying that there was a missing import!
import os, re, math
from pathlib import Path
from datetime import datetime

import dotenv, yaml
dotenv.load_dotenv("configs/local.env")


def model_cache_path(model_id):
    home_dir = Path.home()
    cache_dir = os.environ.get('HF_HUB_CACHE',  home_dir / ".cache" / "huggingface" / "hub")

    model_hf = Path(cache_dir) / ("models--" + model_id.replace("/", "--"))
    model_ref = (model_hf / "refs" / "main").read_text(encoding="utf-8").strip()
    model_path = model_hf / "snapshots" / model_ref

    return model_path
    # BASE_MODEL = "data/huggingface/hub/models--meta-llama--Llama-3.2-1B/snapshots/4e20de362430cd3b72f300e6b0f18e50e7166e08"

In [ ]:
DATASET_NAME = "ed-donner/pricer-data"
#BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
#BASE_MODEL = "meta-llama/Meta-Llama-3.2-3B"
BASE_MODEL = "meta-llama/Meta-Llama-3.2-1B"

PROJECT_NAME = "pricer"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
# RUN_NAME = "v1"

# Run name for saving the model in the hub
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_ID = f"{os.environ['HF_USER']}/{PROJECT_RUN_NAME}"

In [ ]:
# Optonal: offline mode

os.environ['HF_HUB_OFFLINE'] = 'True'
os.environ['WANDB_MODE'] = 'offline'
BASE_MODEL = model_cache_path(BASE_MODEL)

In [ ]:
# Optional: google colab
from google.colab import userdata

os.environ['HF_USER'] = userdata.get('HF_USER')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

In [ ]:
#from huggingface_hub import login
import wandb
from datasets import load_dataset, Dataset, DatasetDict
import matplotlib.pyplot as plt

#login(os.environ['HF_TOKEN'], add_to_git_credential=True)

# Configure Weights & Biases to record against our project
wandb.init(project=PROJECT_NAME, name=RUN_NAME)
# wandb.login(key=os.environ['WANDB_API_KEY'], force=True)

os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" # if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

In [3]:
from tqdm import tqdm
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

⚙️  Running in WANDB offline mode


In [5]:
dataset = load_dataset(DATASET_NAME)
train, test = dataset['train'], dataset['test']

print(test[0])

Using the latest cached version of the dataset since ed-donner/pricer-data couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at data/huggingface/datasets/ed-donner___pricer-data/default/0.0.0/f38df5f756eafe7e331348b26a823e4d706f4a08 (last modified on Mon Aug 11 15:34:31 2025).


{'text': "How much does this cost to the nearest dollar?\n\nOEM AC Compressor w/A/C Repair Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2007 2008 - BuyAutoParts NEW\nAs one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality - tested to meet or exceed OEM specifications Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Included in this K\n\nPrice is $", 'price': 374.41}


In [6]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
     bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
# quant_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

print("named_modules:", list(base_model.named_modules()))
#for name, module in base_model.named_modules():
#    if any(x in name for x in ["proj", "fc"]):
#        print(name)

Memory footprint: 1.0 GB
named_modules: [('', LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  

In [ ]:
# Train parameters

# Hyperparameters for QLoRA
LORA_R = 32
LORA_ALPHA = 64
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1

# Hyperparameters for Training
EPOCHS = 1 # you can do more epochs if you wish, but only 1 is needed - more is probably overkill
BATCH_SIZE = 4 # on an A100 box this can go up to 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Admin config - note that SAVE_STEPS is how often it will upload to the hub
# I've changed this from 5000 to 2000 so that you get more frequent saves
STEPS = 50
SAVE_STEPS = 2000

In [10]:
# First, specify the configuration parameters for LoRA
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [11]:
# Next, specify the general configuration parameters for training
train_parameters = SFTConfig(
    output_dir=Path("data") / PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True, # False for cpu
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb", # if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    dataset_text_field="text",
    save_strategy="steps",
    #hub_strategy="every_save",
    #push_to_hub=True,
    #hub_model_id=HUB_MODEL_ID,
    #hub_private_repo=True,

    completion_only_loss=True,
)

In [12]:
# And now, the Supervised Fine Tuning Trainer will carry out the fine-tuning
# Given these 2 sets of configuration parameters
# The latest version of trl is showing a warning about labels - please ignore this warning
# But let me know if you don't see good training results (loss coming down).

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    args=train_parameters,
)

Tokenizing train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/400000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
print(datetime.now().astimezone().strftime("%FT%T%:z"))

# Fine-tune!
fine_tuning.train()

print(datetime.now().astimezone().strftime("%FT%T%:z"))

In [ ]:
# Push our fine-tuned model to Hugging Face
#fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
#print(f"Saved to the hub: {PROJECT_RUN_NAME}")

wandb.finish()